# SQD + SBD Example: ATP Fragment f14 (72 qubits)

This notebook runs the full SQD pipeline using the SBD GPU solver on a 72-qubit molecular fragment from ATP.

**System**: `atp_0_be2_f14` — 36 orbitals, 36 electrons, 72 qubits

**Data**: ADAPT iteration 50, with both hardware (DD + postselectprep) and MPS (chi=64) bitstrings

## Prerequisites

### 1. NVHPC SDK
By following the command line steps, download from https://developer.nvidia.com/hpc-sdk. This provides `mpicxx` with CUDA support.  This will take a few minutes.  

From the command line:

wget https://developer.download.nvidia.com/hpc-sdk/25.1/nvhpc_2025_251_Linux_x86_64_cuda_12.6.tar.gz

tar xpzf nvhpc_2025_251_Linux_x86_64_cuda_12.6.tar.gz

nvhpc_2025_251_Linux_x86_64_cuda_12.6/install

### 2. LAPACK/BLAS
```bash
# Ubuntu
sudo apt install liblapack-dev libblas-dev
```

### 3. Set GPU Config
Edit sbd_source/app/Configuration

set -gpu=ccXX to match your GPU
cc70=V100, cc80=A100, cc89=RTX 4080/4090, etc

set path to NVHPC root.  Ensure that your version matches the config (IE 25.1)

### 4. Build the SBD executable
```bash
cd sbd_source/app

#Replace the following with your path to MPI bin directory and NVHPC (Make sure version numbers match), for me:

export PATH=/mnt/ffs24/home/rowlan91/ben/f14_example/Linux_x86_64/25.1/compilers/bin:$PATH

export PATH=/mnt/ffs24/home/rowlan91/ben/f14_example/Linux_x86_64/25.1/comm_libs/mpi/bin:$PATH

make
```
Note that there are a lot of warnings, it's okay.

### 5. Python dependencies
```bash
pip install numpy qiskit-addon-sqd
```

In [4]:
import collections
import os
import pickle
import re
import subprocess
import tempfile
import time

import numpy as np
from qiskit_addon_sqd.subsampling import postselect_by_hamming_right_and_left, subsample

# System parameters
NORB = 36
NELEC = 36
NQUBITS = 2 * NORB
N_ALPHA = NELEC // 2
N_BETA = NELEC // 2

# Path to built SBD executable
SBD_EXE = "sbd_source/app/diag"
FCIDUMP = "data/hamiltonians/atp_0_be2_f14.fcidump"

# Paths to MPI 
os.environ["MPIRUN"] = "/mnt/ffs24/home/rowlan91/ben/f14_example/Linux_x86_64/25.1/comm_libs/hpcx/bin/mpirun"

os.environ["OMPI_MCA_rmaps_base_oversubscribe"] = "1"
os.environ["OMPI_MCA_hwloc_base_binding_policy"] = "none"

## Step 1: Load the data

We have two sources of bitstrings for ADAPT iteration 50:
- **Hardware DD**: 389k shots from `ibm_boston` with dynamical decoupling + post-select prep
- **MPS chi=64**: 10k classically-simulated shots (23 unique bitstrings, 100% correct electron count)

The MPS data is in a two-register format (`"72bits 72zeros"`) — we strip the zero-padding.

In [5]:
# Load hardware DD counts
with open("data/counts/hardware_dd.pkl", "rb") as f:
    hw_counts = pickle.load(f)
print(f"Hardware DD: {len(hw_counts):,} unique, {sum(hw_counts.values()):,} shots")

# Load MPS counts and strip zero-padding
with open("data/counts/mps_chi64.pkl", "rb") as f:
    mps_raw = pickle.load(f)

mps_counts = {}
for bs, count in mps_raw.items():
    clean = bs.split(" ")[0]  # keep first 72 bits, discard 72 zeros
    mps_counts[clean] = mps_counts.get(clean, 0) + count
print(f"MPS chi=64:  {len(mps_counts):,} unique, {sum(mps_counts.values()):,} shots")

# Merge into one pool
merged = dict(hw_counts)
for bs, count in mps_counts.items():
    merged[bs] = merged.get(bs, 0) + count
print(f"Merged:      {len(merged):,} unique, {sum(merged.values()):,} shots")

# Save merged counts for run_sqd_sbd.py
# The script expects counts at data/results/atp_0_be2_f14/{fragment}_{adapt}_adaptiterations.qasm_counts_...
os.makedirs("data/results/atp_0_be2_f14", exist_ok=True)
merged_path = "data/results/atp_0_be2_f14/atp_0_be2_f14_050_adaptiterations.qasm_counts_merged"
with open(merged_path, "wb") as f:
    pickle.dump(merged, f)
print(f"\nSaved merged counts to: {merged_path}")

Hardware DD: 389,228 unique, 389,228 shots
MPS chi=64:  23 unique, 10,000 shots
Merged:      389,251 unique, 399,228 shots

Saved merged counts to: data/results/atp_0_be2_f14/atp_0_be2_f14_050_adaptiterations.qasm_counts_merged


## Step 2: Run SQD+SBD

We call `run_sqd_sbd.py` which handles the full pipeline.

The script takes the fragment name, paths to circuits/hamiltonians/results, and the SBD executable as arguments.

In [6]:
import subprocess
import sys

# Output directory for results
os.makedirs("output", exist_ok=True)

cmd = [
    sys.executable, "run_sqd_sbd.py",
    "--fragment", "atp_0_be2_f14",
    "--circuit_dir", "data/circuits",
    "--hamiltonian_dir", "data/hamiltonians",
    "--results_dir", "data/results",
    "--sbd_exe", SBD_EXE,
    "--output_dir", "output",
    "--max_iterations", "100",
    "--energy_tol", "1e-6",
    "--occupancies_tol", "1e-6",
    "--num_gpus", "1",
    "50",  # ADAPT iteration
]

print("Command:")
print(" ".join(cmd))
print()

# Run with live output
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\nExit code: {process.returncode}")

Command:
/mnt/home/rowlan91/.conda/envs/thesisEnv/bin/python run_sqd_sbd.py --fragment atp_0_be2_f14 --circuit_dir data/circuits --hamiltonian_dir data/hamiltonians --results_dir data/results --sbd_exe sbd_source/app/diag --output_dir output --max_iterations 100 --energy_tol 1e-6 --occupancies_tol 1e-6 --num_gpus 1 50

Reading Hamiltonian...
Parsing data/hamiltonians/atp_0_be2_f14.fcidump
  NORB=36, NELEC=36, ECORE=0.0
Loading measurement data...
  ADAPT iter 50: 778502 raw unique, 798456 shots (2 files) -> 389251 merged unique
  Total: 389251 unique bitstrings, 798456 shots
  Raw bitstring matrix: (389251, 72)

Starting SQD loop: 2 batches, 500 samples/batch
Max iterations: 100

--- SQD Iteration 1 ---
  After postselect/recovery: 7251 unique bitstrings (0.0s)
  Batch 0: 996 unique CI strings (992016 determinants)
    [SBD] Elapsed time for helper construction 0.09490899999999999 (sec)
    [SBD] Elapsed time for init 0.002525 (sec)
    [SBD] Elapsed time for diagonal H (GPU) 0.0091869

KeyboardInterrupt: 